## 1. Read fines.csv

In [1]:
import pandas as pd
import gc

df = pd.read_csv("../data/fines.csv")
df

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2.0,3200.0,Ford,Focus,1989
1,E432XX77RUS,1.0,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1.0,2100.0,Ford,Focus,1984
3,X582HE161RUS,2.0,2000.0,Ford,Focus,2015
4,92918M178RUS,1.0,5700.0,Ford,Focus,2014
...,...,...,...,...,...,...
925,A111AA77RUS,1.0,1500.0,Ford,Focus,2005
926,B222BB77RUS,2.0,3200.0,Toyota,Corolla,2010
927,C333CC77RUS,1.0,7000.0,BMW,X5,2018
928,D444DD77RUS,0.0,5000.0,Audi,A4,2012


## 2. Iterations

### - for + iloc

In [2]:
%%timeit
res = []
for i in range(len(df)):
    res.append(df.iloc[i]["Fines"] / df.iloc[i]["Refund"] * df.iloc[i]["Year"]
    if df.iloc[i]["Refund"] != 0 else 0)

df["calc"] = res

142 ms ± 17.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### - iterrows()

In [3]:
%%timeit
res = []
for _, row in df.iterrows():
    if row["Refund"] != 0:
        res.append(row["Fines"] / row["Refund"] * row["Year"])
    else:
        res.append(0)
df["calc"] = res

39.8 ms ± 4.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### - apply()

In [4]:
%%timeit
df["calc"] = df.apply(
    lambda r: r["Fines"] / r["Refund"] * r["Year"] if r["Refund"] != 0 else 0,
    axis=1
)

12.8 ms ± 977 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### - Vectorized operation

In [5]:
%%timeit
df["calc"] = df["Fines"] / df["Refund"] * df["Year"]
df["calc"] = df["calc"].fillna(0)

348 μs ± 23 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### - .values

In [6]:
%%timeit
df["calc"] = df["Fines"].values / df["Refund"].values * df["Year"].values

<magic-timeit>:1: RuntimeWarning: divide by zero encountered in divide


103 μs ± 7.98 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 3. Indexing 


### - Measure lookup without index

In [7]:
%%timeit
df[df["CarNumber"] == "O136HO197RUS"]

356 μs ± 50.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### - Set CarNumber as index

In [8]:
df = df.set_index("CarNumber")

### Measure lookup with index

In [9]:
%%timeit
df.loc["O136HO197RUS"]

78.3 μs ± 2.22 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 4. Downcasting 

In [10]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE77RUS
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Refund  930 non-null    float64
 1   Fines   930 non-null    float64
 2   Make    930 non-null    object 
 3   Model   919 non-null    object 
 4   Year    930 non-null    int64  
 5   calc    930 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 236.0 KB


### - Downcast numeric types

In [11]:
df["Fines"] = pd.to_numeric(df["Fines"], downcast="float")
df["Refund"] = pd.to_numeric(df["Refund"], downcast="float")
df["Year"] = pd.to_numeric(df["Year"], downcast="integer")

In [12]:
df["Make"] = df["Make"].astype("category")
df["Model"] = df["Model"].astype("category")

In [13]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE77RUS
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   Refund  930 non-null    float32 
 1   Fines   930 non-null    float32 
 2   Make    930 non-null    category
 3   Model   919 non-null    category
 4   Year    930 non-null    int16   
 5   calc    930 non-null    float64 
dtypes: category(2), float32(2), float64(1), int16(1)
memory usage: 114.8 KB


## Final step

In [14]:
%reset_selective df
import gc
gc.collect()



Nothing done.


69